# Insight 1-1. Spot-Futures 베이시스 PoC

같은 심볼의 **현물(Spot) 가격**과 **무기한 선물(Perpetual) 마크 가격**이 벌어지는 정도(베이시스)를
실시간으로 계산하고, 통계적 임계치 + 지속성 조건을 적용해 과열 신호를 판정한다.

- 관련 문서: [APIdocs/analyticsInsightsREADME.md — §1. 실시간 차익거래 기회 및 스프레드 탐지](../APIdocs/analyticsInsightsREADME.md#1-실시간-차익거래-기회-및-스프레드-탐지)
  안의 **Insight 1-1** 항목 (헤딩이 아니라 불릿이라 직접 앵커는 안 걸린다)
- 운영 파이프라인에서는 이 조인/판정을 Kafka Streams `KStream-KStream` 윈도우 조인으로 구현하지만,
  이 PoC는 그 로직을 하나의 프로세스 안에서 **두 웹소켓의 최신값을 유지 + 주기적 샘플링**으로
  단순화해 흉내낸다.
- **알려진 네트워크 이슈**: `fstream.binance.com`(USD-M 선물) 웹소켓은 네트워크/방화벽에 따라
  연결은 되지만 메시지가 오지 않는 경우가 있다([usdm_futures_market_stream.ipynb](../APIpoc/usdm_futures_market_stream.ipynb) 참고).
  이 노트북은 그 경우 자동으로 REST 폴링(`GET /fapi/v1/premiumIndex`)으로 전환한다.

In [ ]:
%pip install -q pandas websocket-client

In [2]:
import json
import statistics
import threading
import time
import urllib.request
import uuid
from collections import deque

import websocket

SYMBOL = "btcusdt"

SPOT_WS = "wss://stream.binance.com:9443/stream"
FUTURES_WS = "wss://fstream.binance.com/market/stream"
FUTURES_REST = f"https://fapi.binance.com/fapi/v1/premiumIndex?symbol={SYMBOL.upper()}"

# analyticsInsightsREADME.md Insight 1-1의 판정 기준값
WINDOW_SECONDS = 5          # basis 샘플링 주기 (문서: 5~10초 tumbling/hopping window)
STATS_WINDOW = 20           # 이동평균/표준편차에 쓸 최근 윈도우 개수 (문서: 20~30개)
Z_THRESHOLD = 2.0           # |z| > 2 를 과매수/과매도 후보로 판정
CONSECUTIVE_REQUIRED = 3    # 연속 3~5개 윈도우 유지돼야 알림 (문서: 3~5개)
MIN_SAMPLES_FOR_STATS = 5   # 표준편차가 아직 불안정한 초반 구간은 z-score를 계산하지 않는다
RUN_SECONDS = 300           # 총 수집 시간 (5분). STATS_WINDOW를 채우려면 최소 WINDOW_SECONDS*STATS_WINDOW초 필요

def ws_subscribe(base_url: str, stream: str, timeout: float = 10):
    """base_url에 연결하고 SUBSCRIBE 메시지를 보낸 뒤 ack까지 확인한 연결을 반환한다."""
    ws = websocket.create_connection(base_url, timeout=timeout)
    ws.send(json.dumps({"method": "SUBSCRIBE", "params": [stream], "id": str(uuid.uuid4())}))
    ack = json.loads(ws.recv())
    print(f"subscribe ack ({stream}):", ack)
    return ws


def peek(stream: str, base_url: str, duration: float = 10, timeout: float = 10) -> list[dict]:
    messages = []
    ws = ws_subscribe(base_url, stream, timeout=timeout)
    end_time = time.time() + duration
    try:
        while time.time() < end_time:
            try:
                raw = ws.recv()
            except websocket.WebSocketTimeoutException:
                print(f"{timeout}초 동안 응답 없음: {len(messages)}건 수신")
                break
            msg = json.loads(raw)
            if msg.get("stream") == stream:
                messages.append(msg["data"])
    finally:
        ws.close()
    return messages

In [10]:
SPOT_STREAM = f"{SYMBOL}@bookTicker"

duration = 10
bookticker_sample = peek(SPOT_STREAM, SPOT_WS, duration=duration)

print("bookTicker sample :", bookticker_sample)
print("bookTicker sample count :", len(bookticker_sample))
print("1s traffic rate (bookTicker):", len(bookticker_sample) / duration, "msg/sec")

subscribe ack (btcusdt@bookTicker): {'result': None, 'id': '63be3d73-c561-4824-9445-db738b4ef22f'}
bookTicker sample : [{'u': 99544785717, 's': 'BTCUSDT', 'b': '77270.00000000', 'B': '5.22862000', 'a': '77270.01000000', 'A': '8.41793000'}, {'u': 99544785730, 's': 'BTCUSDT', 'b': '77270.00000000', 'B': '5.22862000', 'a': '77270.01000000', 'A': '8.41786000'}, {'u': 99544785757, 's': 'BTCUSDT', 'b': '77270.00000000', 'B': '5.22862000', 'a': '77270.01000000', 'A': '8.41139000'}, {'u': 99544785777, 's': 'BTCUSDT', 'b': '77270.00000000', 'B': '5.22862000', 'a': '77270.01000000', 'A': '8.41146000'}, {'u': 99544785781, 's': 'BTCUSDT', 'b': '77270.00000000', 'B': '5.22862000', 'a': '77270.01000000', 'A': '8.41793000'}, {'u': 99544785782, 's': 'BTCUSDT', 'b': '77270.00000000', 'B': '5.22862000', 'a': '77270.01000000', 'A': '8.41786000'}, {'u': 99544785795, 's': 'BTCUSDT', 'b': '77270.00000000', 'B': '5.22862000', 'a': '77270.01000000', 'A': '8.41139000'}, {'u': 99544785796, 's': 'BTCUSDT', 'b': 

In [11]:
FUTURES_STREAM = f"{SYMBOL}@markPrice@1s"

# btcusdt @ markPrice — 10초 동안 스트림을 수신해 파싱
# 필요에 맞게 파싱 로직 커스터마이즈 (지금은 parse_mark_price 사용)
markprice_sample = peek(FUTURES_STREAM, FUTURES_WS, duration=10)
print("markPrice sample :", markprice_sample)
print("markPrice sample count :", len(markprice_sample))
print("1s traffic rate (markPrice):", len(markprice_sample) / duration, "msg/sec")

subscribe ack (btcusdt@markPrice@1s): {'result': None, 'id': 'ec36bee1-de88-4856-a100-1b976e4e2f37'}
markPrice sample : [{'e': 'markPriceUpdate', 'E': 1788312434000, 's': 'BTCUSDT', 'p': '77220.41461594', 'ap': '77220.41461594', 'P': '77255.52790080', 'i': '77260.02608696', 'r': '0.00007980', 'T': 1788336000000, 'st': 1}, {'e': 'markPriceUpdate', 'E': 1788312435000, 's': 'BTCUSDT', 'p': '77220.41461594', 'ap': '77220.41461594', 'P': '77255.52438394', 'i': '77260.02608696', 'r': '0.00007980', 'T': 1788336000000, 'st': 1}, {'e': 'markPriceUpdate', 'E': 1788312436002, 's': 'BTCUSDT', 'p': '77220.41896377', 'ap': '77220.41896377', 'P': '77255.52208249', 'i': '77260.03043478', 'r': '0.00007980', 'T': 1788336000000, 'st': 1}, {'e': 'markPriceUpdate', 'E': 1788312437000, 's': 'BTCUSDT', 'p': '77220.41896377', 'ap': '77220.41896377', 'P': '77255.52208249', 'i': '77260.03043478', 'r': '0.00007980', 'T': 1788336000000, 'st': 1}, {'e': 'markPriceUpdate', 'E': 1788312438002, 's': 'BTCUSDT', 'p': '

## 1. 최신 spot / futures 가격 유지 (KTable materialize 역할)

운영 파이프라인의 `KStream-KStream` 윈도우 조인 대신, 두 웹소켓이 각자 들어오는 대로 `latest`
딕셔너리를 최신값으로 갱신한다. `listen_futures`는 먼저 웹소켓을 시도하고, 연결은 되지만 일정
시간(10초) 동안 메시지가 안 오면 REST 폴링(`GET /fapi/v1/premiumIndex`)으로 자동 전환한다 —
[collectorInsightsREADME.md](../APIdocs/collectorInsightsREADME.md)에 나온 "웹소켓이 막히는
환경을 가정한 REST 폴링 대체 경로" 패턴과 같은 접근이다.

In [ ]:
latest = {
    "spot_mid": None,
    "spot_ts": None,
    "mark_price": None,
    "funding_rate": None,
    "futures_ts": None,
    "futures_source": None,
}

stop_event = threading.Event()


def listen_spot():
    while not stop_event.is_set():
        try:
            ws = ws_subscribe(SPOT_WS, SPOT_STREAM, timeout=10)
        except Exception as e:
            print(f"[spot] 연결 실패({e!r}), 3초 후 재시도")
            time.sleep(3)
            continue

        try:
            while not stop_event.is_set():
                raw = ws.recv()
                msg = json.loads(raw)
                if msg.get("stream") != SPOT_STREAM:
                    continue
                data = msg["data"]
                bid, ask = float(data["b"]), float(data["a"])
                latest["spot_mid"] = (bid + ask) / 2
                latest["spot_ts"] = time.time()
        except Exception as e:
            print(f"[spot] websocket 끊김({e!r}), 재연결")
        finally:
            ws.close()


def get_json(url: str) -> dict:
    with urllib.request.urlopen(url, timeout=10) as resp:
        return json.load(resp)


def listen_futures():
    try:
        # 연결은 됐지만 메시지가 안 오는 경우까지 잡으려면 recv 자체에도
        # 타임아웃을 걸어야 한다 (create_connection의 timeout은 소켓 전체에 적용된다).
        ws = ws_subscribe(FUTURES_WS, FUTURES_STREAM, timeout=10)
        latest["futures_source"] = "websocket"
        while not stop_event.is_set():
            raw = ws.recv()
            msg = json.loads(raw)
            if msg.get("stream") != FUTURES_STREAM:
                continue
            data = msg["data"]
            latest["mark_price"] = float(data["p"])
            latest["funding_rate"] = float(data["r"])
            latest["futures_ts"] = time.time()
        ws.close()
        return
    except Exception as e:
        print(f"[futures] websocket 실패({e!r}), REST 폴링으로 전환")

    latest["futures_source"] = "rest-poll"
    while not stop_event.is_set():
        try:
            data = get_json(FUTURES_REST)
            latest["mark_price"] = float(data["markPrice"])
            latest["funding_rate"] = float(data["lastFundingRate"])
            latest["futures_ts"] = time.time()
        except Exception as e:
            print(f"[futures] REST 폴링 오류: {e!r}")
        time.sleep(1)

## 2. Basis 계산 + 통계적 임계치 + 지속성 조건 (Insight 1-1 판정 로직)

[analyticsInsightsREADME.md](../APIdocs/analyticsInsightsREADME.md)의 "과매도 / 과매수 판단
기준"과 "방식"의 펀딩비 필터를 그대로 옮긴다.

1. **통계적 임계치**: 최근 `STATS_WINDOW`개 basis 값의 이동평균(μ)·표준편차(σ)로
   `z = (basis - μ) / σ`를 구해 `|z| > Z_THRESHOLD`면 후보로 판정한다.
2. **펀딩비 방향 일치 필터**: "베이시스와 펀딩비(`r`)를 같이 겹쳐서, 베이시스 확대 방향과
   펀딩비 부호가 일치하는 구간만 알림 대상으로 좁히면 노이즈가 줄어든다" — `z`와 `fundingRate`의
   부호가 같을 때만(`z>0`이면 콘탱고 확대+롱 과열, `z<0`이면 백워데이션 확대+숏 과열) 후보로 카운트한다.
3. **시간 윈도우 유지 조건**: 위 두 조건을 동시에 만족하는 상태가 `CONSECUTIVE_REQUIRED`개 윈도우
   연속으로 유지될 때만 알림을 1회 발행한다(디바운스). 조건이 풀리면 카운터를 0으로 리셋한다.

In [ ]:
basis_history: deque[float] = deque(maxlen=STATS_WINDOW)
consecutive_count = 0
records: list[dict] = []


def sample_once():
    global consecutive_count

    if latest["spot_mid"] is None or latest["mark_price"] is None or latest["funding_rate"] is None:
        return None

    basis = latest["mark_price"] - latest["spot_mid"]
    basis_history.append(basis)

    mean = statistics.mean(basis_history)
    std = statistics.pstdev(basis_history) if len(basis_history) >= 2 else 0.0

    if len(basis_history) >= MIN_SAMPLES_FOR_STATS and std > 0:
        z = (basis - mean) / std
    else:
        z = 0.0

    over_threshold = abs(z) > Z_THRESHOLD
    # 베이시스 확대 방향(z 부호)과 펀딩비 부호가 일치하는 구간만 후보로 카운트한다.
    funding_aligned = (z > 0 and latest["funding_rate"] > 0) or (z < 0 and latest["funding_rate"] < 0)
    candidate = over_threshold and funding_aligned

    consecutive_count = consecutive_count + 1 if candidate else 0
    alert = consecutive_count == CONSECUTIVE_REQUIRED

    record = {
        "ts": latest["futures_ts"],
        "spot_mid": latest["spot_mid"],
        "mark_price": latest["mark_price"],
        "funding_rate": latest["funding_rate"],
        "basis": basis,
        "mean": mean,
        "std": std,
        "z": z,
        "funding_aligned": funding_aligned,
        "consecutive_count": consecutive_count,
        "alert": alert,
        "futures_source": latest["futures_source"],
    }
    records.append(record)
    return record

## 3. 실행 — 실시간 수집 + 판정

`RUN_SECONDS`(기본 5분) 동안 `WINDOW_SECONDS`마다 샘플링한다. 알림 조건
(`consecutive_count == CONSECUTIVE_REQUIRED`)에 걸리면 `ALERT`를 출력한다.
`STATS_WINDOW`(20개)를 채우기 전까지는 z-score가 불안정하므로 초반
`WINDOW_SECONDS * MIN_SAMPLES_FOR_STATS`초 정도는 워밍업 구간으로 본다.

In [ ]:
def run(run_seconds: int):
    stop_event.clear()
    spot_thread = threading.Thread(target=listen_spot, daemon=True)
    futures_thread = threading.Thread(target=listen_futures, daemon=True)
    spot_thread.start()
    futures_thread.start()

    time.sleep(2)  # 첫 메시지 도착 대기

    end = time.time() + run_seconds
    while time.time() < end:
        rec = sample_once()
        if rec:
            flag = " <-- ALERT" if rec["alert"] else ""
            print(
                f"basis={rec['basis']:+.3f} mean={rec['mean']:+.3f} std={rec['std']:.3f} "
                f"z={rec['z']:+.2f} consec={rec['consecutive_count']}{flag} "
                f"(source={rec['futures_source']})"
            )
        else:
            print("데이터 대기 중...")
        time.sleep(WINDOW_SECONDS)

    stop_event.set()
    spot_thread.join(timeout=5)
    futures_thread.join(timeout=5)
    print(f"\ntotal records: {len(records)}")


run(RUN_SECONDS)

## 4. 결과 확인

수집된 레코드를 `DataFrame`으로 정리해서 basis/z-score 추이와 실제 알림 발생 지점을 확인한다.

In [ ]:
import pandas as pd

df = pd.DataFrame(records)
df.tail(10)

In [ ]:
# 알림이 발생한 시점만 모아보기
df[df["alert"]]

## 참고

- 여기서는 `basis = markPrice - spotMidPrice`를 썼다 — 바이낸스 자체 spot 기준으로 실행
  가능한 차익거래 신호를 잡기 위함이며, 여러 거래소 평균인 `indexPrice`를 쓰지 않는 이유는
  실제로 체결 가능한 가격이 아니기 때문이다.
- 운영 환경으로 옮길 때는 이 in-memory 최신값 유지 방식을 Kafka Streams
  `KStream-KStream` 윈도우 조인(5~10초 tumbling/hopping, key=`symbol`)으로 교체하고,
  알림 상태(`consecutive_count`)는 `KTable`의 상태 저장소에 둔다
  ([collectorInsightsREADME.md §7](../APIdocs/collectorInsightsREADME.md#7-basis베이시스-조인--spot-trade--usd-m-markprice) 참고).